In [1]:
import os
from distutils.command.upload import upload

from IPython.terminal.shortcuts.auto_suggest import accept
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

####  · SystemMessage
####  · HumanMessage
####  · AIMessage
####  · ToolMessage


In [2]:
from langchain_core.messages import SystemMessage
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """
    Get the weather for a location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny!"

agent = create_agent(model="deepseek-chat", tools=[get_weather])

response = agent.invoke({
    "messages" : [
        SystemMessage(content="Please use tools to get the weather."),
        HumanMessage(content="What's the weather like in San Francisco?"),
    ]
})

print(response)

{'messages': [SystemMessage(content='Please use tools to get the weather.', additional_kwargs={}, response_metadata={}, id='85a5cc09-495a-4e0d-9e87-6473e801acc8'), HumanMessage(content="What's the weather like in San Francisco?", additional_kwargs={}, response_metadata={}, id='09152ad4-72ce-421d-af31-193d2c26b881'), AIMessage(content="I'll check the weather in San Francisco for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 293, 'total_tokens': 342, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 293}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '4be75f46-b4f6-40e5-a0f0-43f3a8146cb1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a18

In [3]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

Please use tools to get the weather.
================================ Human Message =================================

What's the weather like in San Francisco?
================================== Ai Message ==================================

I'll check the weather in San Francisco for you.
Tool Calls:
  get_weather (call_00_HKFR2a2XmhD4W0d24z1B6591)
 Call ID: call_00_HKFR2a2XmhD4W0d24z1B6591
  Args:
    location: San Francisco
================================= Tool Message =================================
Name: get_weather

Current weather in San Francisco is sunny!
================================== Ai Message ==================================

The weather in San Francisco is currently **sunny** ☀️. Enjoy the nice weather!


# 2.多模态消息

## 2.1.在线图片

In [4]:
# 原始方法
# message = {
#     "role" : "user",
#     "content" : [
#         {"type" : "text", "text" : "描述一下图片信息。"},
#         {"type" : "image", "url" : "https://webstatic.mihoyo.com/upload/contentweb/2022/10/20/7dc2db8ab2436b148a13179db41d4397_2851152408022290858.jpg"}
#     ]
# }

# 新方法
message = HumanMessage(
    content=[
        {"type": "text", "text": "描述一下图片信息。"},
        {"type": "image", "url": "https://webstatic.mihoyo.com/upload/contentweb/2022/10/20/7dc2db8ab2436b148a13179db41d4397_2851152408022290858.jpg"}
    ]
)

In [5]:
# agent.stream返回的是一个生成器
stream = agent.stream(
    {"messages" : [message]},
    stream_mode="messages"
)

for chunk,metadata in stream:
    print(chunk.content, end="", flush=True)

这张图片是游戏《原神》（右上角有着“原神”的标志，左下角有“© miHoYo”版权字样）的宣传海报，展现了多位各具特色的角色（主要由须弥地区的角色组成），整体色调以蓝绿、白、黑和金色为主，充满奇幻色彩和动感。以下是主要元素的描述：

**前景核心角色：**
*   **左侧（散兵/流浪者）：** 占据画面左半部分，他戴着标志性的巨大斗笠，身穿深蓝与黑色为主的服饰，配有红色系带和金色流苏。他有着深蓝色的短发和紫色的眼睛，表情平静中透着自信与一丝桀骜。
*   **右侧（纳西妲）：** 画面右半部分是一位白发绿瞳的精灵少女，她闭着双眼，神态恬静优雅。她身穿白色与绿色相间的服饰，散发着温柔的光芒。

**中景与背景角色（从右上、左侧到下方）：**
*   **左后方边缘：** 一位戴着半遮面面具的长发角色（魈），身披深色羽衣，配有绿色发光的神之眼和流苏。
*   **中央偏上：** 金色短发的少年（空）作为旅行者在中央位置，身穿黑金色服饰，正侧身回望。
*   **空的身后（左上方）：** 漂浮着一位小巧的白色飞行向导（派蒙）。
*   **空的上方：** 一位银灰色头发、身穿青绿色与黑色相间服饰的男性角色（艾尔海森）正持剑而立。
*   **右上角：** 一位身材高挑、深色皮肤的女性角色（迪希雅），身穿黑色与金色服饰，手持一把巨大的巨剑，姿态充满力量感。
*   **右下方（纳西妲身前）：** 一位深蓝色头发、穿着蓝色露脐装的少女（莱依拉），以及她身旁一位戴着兜帽、身穿蓝紫色服饰的女性角色（坎蒂丝），手持盾牌与长矛。

**其他细节：**
*   画面中穿插着各种武器（如单手剑、巨剑、长柄武器）和元素特效（如青色的风、绿色的草元素、金色的光芒）。
*   整体构图层次丰富，人物错落有致，体现了群像的宏大与角色间的羁绊。

## 2.2.本地图片

In [6]:
# 模拟上传窗口
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [7]:
# 读取图片，转为base64字符串
import base64

# 获取第一个也就是唯一一个上传的文件
upload_file = uploader.value[0]

# 获取其内存视图
content_mv = upload_file['content']

# 转换内存视图->字节
img_byte = bytes(content_mv)

# base64编码
img_b64 = base64.b64encode(img_byte).decode("utf-8")

In [8]:
multimodal_question = HumanMessage(content=[
    {
        "type" : "image",
        "base64" : img_b64,
        "mime_type" : "image/jpeg",
    },
    {"type" : "text", "text" : "分析一下这张图片"}
])

for chunk, metadata in agent.stream(
    {"messages" : [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)

这是一张非常典型的**科幻/未来主义风格**的场景截图，结合画面下方的UI信息，可以确认这是出自热门3D游戏**《明日方舟：终末地》（Arknights: Endfield）**的游戏画面。

以下是对这张图片的详细分析：

### 1. 场景与空间设计
*   **空间形态**：画面呈现一个巨大的、呈弧形的室内空间（类似指挥中心、大厅或空间站内部）。顶部有多层环形的巨大结构，带有流线型的照明灯带，营造出极度宏大、开阔的工业科幻感。
*   **配色与光影**：主色调为冷峻的深灰、黑色与白色，搭配极具视觉冲击力的**亮橙色**（用于主屏幕边框、漂浮的光点以及树木）。环境光线偏暗，主要光源来自顶部的灯带和中央巨大的发光屏幕，形成了强烈的明暗对比。
*   **细节元素**：屏幕两侧有类似空间站或机械结构的线稿背景图；场景中点缀着发光的橙色粒子（类似萤火虫或数据流）；下方平台两侧各有一棵充满科技感的“橙色发光树”（可能是全息投影或人造植物）；下方还有微缩的人形（玩家角色/工作人员）在走动或交谈，通过人物比例反衬出建筑的巨大。

### 2. 核心视觉焦点（中央大屏幕）
*   **文字内容**：屏幕中央用醒目的白色字体写着**“[勇于探索 叩问苍穹]”**。这是一句极具航天精神与探索精神的标语。
*   **Logo**：文字上方有一个带有橙色圆环的白色箭头/塔状标志，这通常是游戏内特定阵营、组织或空间站的徽标。
*   **VI设计**：大屏幕采用黑底橙边，文字下方带有橙色发光条和网格纹理，屏幕两侧的线框图增加了工业技术感。

### 3. 画面外信息（UI与元数据）
*   **左下角**：“千重子 1629866593” —— 这可能是该游戏截图者的**游戏内昵称（ID）**以及UID。
*   **右下角**：
    *   **(f/32) 16 mm**：模拟摄影参数。f/32 是非常小的光圈（通常用于保持极大景深，让远近景物都清晰），16mm 是广角镜头，这解释了为什么画面能囊括如此宏大的空间结构，并带有一定的透视畸变。这是《终末地》游戏内置的拍照/摄影系统的一大特色。
    *   **“明日方舟 终末地 (ARKNIGHTS: ENDFIELD)”**：官方Logo，直接点名了游戏出处。

### 4. 风格与背景推测
*   **美术风格**：延